# PNU data

In [30]:
import os, sys, math, time, logging
import requests
import pandas as pd
from tqdm import tqdm
import geopandas as gpd

In [31]:
# Read the shapefile
gdf = gpd.read_file(r"D:\000_SCI\7_Typology_Pattern\6_analysis_GIS_data\5_final\GMM_clustered_4.shp", encoding="UTF-8")

# Quick overview
print(gdf.shape)
print(gdf.dtypes)
print(gdf.head())
print(gdf.crs)

(2292, 57)
plot_area_     float64
households     float64
apt_code        object
A1              object
202301_x       float64
202302_x       float64
202303_x       float64
202304_x       float64
202305_x       float64
202306_x       float64
202307_x       float64
202308_x       float64
202309_x       float64
202310_x       float64
202311_x       float64
202312_x       float64
spring_x        object
summer_x        object
autumn_x        object
winter_x        object
total_x         object
city            object
bldg_no          int64
CR             float64
bldg_heigh     float64
bldg_bo        float64
bldg_blw       float64
bldg_bc        float64
bldg_age       float64
pop_densit     float64
land_price       int64
apt_lw         float64
apt_orien      float64
apt_elev       float64
plot_are_1     float64
floors         float64
202301_y         int64
202302_y         int64
202303_y         int64
202304_y         int64
202305_y         int64
202306_y         int64
202307_y         int64


In [33]:
df = pd.DataFrame()
df["PNU"] = gdf["A1"].astype(str).str.strip().str.zfill(19)

# Correct slicing based on verified example: 1168010300100120004
df["sigunguCd"] = df["PNU"].str[0:5]    # 5 digits → 11680
df["bjdongCd"]  = df["PNU"].str[5:10]   # 5 digits → 10300
df["platGbCd"]  = df["PNU"].str[10:11]  # 1 digit  → 1
df["bun"]       = df["PNU"].str[11:15]  # 4 digits → 0012
df["ji"]        = df["PNU"].str[15:19]  # 4 digits → 0004

# ── 3. Verify with known example ─────────────────────────────────
print("\n── Verification with known PNU: 1168010300100120004 ──")
test = "1168010300106510001"
print(f"  sigunguCd : {test[0:5]}   (expected: 11680)")
print(f"  bjdongCd  : {test[5:10]}  (expected: 10300)")
print(f"  platGbCd  : {test[10:11]}       (expected: 1)")
print(f"  bun       : {test[11:15]}  (expected: 0012)")
print(f"  ji        : {test[15:19]}  (expected: 0004)")


── Verification with known PNU: 1168010300100120004 ──
  sigunguCd : 11680   (expected: 11680)
  bjdongCd  : 10300  (expected: 10300)
  platGbCd  : 1       (expected: 1)
  bun       : 0651  (expected: 0012)
  ji        : 0001  (expected: 0004)


In [29]:

# ── 7. Save ───────────────────────────────────────────────────────
out_cols = ["sigunguCd", "bjdongCd", "bun", "ji", "PNU", "platGbCd", "guName"]
out_path = r"D:\5-Data\DATA\필지\seoul_merge\parcels_for_energy.csv"
df[out_cols].to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n✓ Saved {len(df):,} parcels → {out_path}")



✓ Saved 902,001 parcels → D:\5-Data\DATA\필지\seoul_merge\parcels_for_energy.csv


In [30]:
# ── 8. Summary ────────────────────────────────────────────────────
print("\nParcels per gu:")
print(df.groupby("guName").size().sort_values(ascending=False).to_string())

print("\nplatGbCd breakdown:")
print(df["platGbCd"].value_counts().rename({
    "0": "0=대지", "1": "1=산", "2": "2=블록"
}).to_string())

print("\nSample output:")
print(df[out_cols].head(5).to_string(index=False))


Parcels per gu:
guName
성북구     53004
종로구     49087
은평구     47517
관악구     45272
용산구     44468
강서구     43789
강북구     40323
동대문구    40301
마포구     39815
중랑구     38647
서대문구    38616
동작구     38235
영등포구    36929
구로구     36206
서초구     35142
강남구     34489
중구      34248
광진구     32579
송파구     31300
강동구     30960
성동구     26505
도봉구     22542
양천구     21626
노원구     20562
금천구     19839

platGbCd breakdown:
platGbCd
1=산     885042
2=블록     16959

Sample output:
sigunguCd bjdongCd  bun   ji                 PNU platGbCd guName
    11680    10100 0601 0000 1168010100106010000        1    강남구
    11680    10100 0601 0001 1168010100106010001        1    강남구
    11680    10100 0601 0002 1168010100106010002        1    강남구
    11680    10100 0601 0003 1168010100106010003        1    강남구
    11680    10100 0601 0004 1168010100106010004        1    강남구


# API download

In [19]:
"""
================================================================================
Seoul Building Energy Downloader — Gu by Gu
================================================================================

For each gu (25 gu in Seoul), in order:

  ELECTRICITY:
    Step A. Scan January → find parcels with electricity data in this gu
    Step B. Download Feb–Dec for those active parcels
    Step C. Save → electricity_{year}_{guName}.csv

  GAS:
    Step D. Scan January → find parcels with gas data in this gu
    Step E. Download Feb–Dec for those active parcels
    Step F. Save → gas_{year}_{guName}.csv

  → Move to next gu

At the end, merge all gu files into:
    electricity_{year}_seoul.csv
    electricity_{year}_seoul_panel.csv
    gas_{year}_seoul.csv
    gas_{year}_seoul_panel.csv

RESUME:
    Completed gu are skipped automatically on re-run.
    If interrupted mid-gu, resumes from that gu's last position.

SETUP:
    pip install requests pandas tqdm
    python seoul_energy_download.py
================================================================================
"""

import os, sys, math, time, json, logging, smtplib, traceback
from datetime import date, datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import requests
import pandas as pd
from tqdm import tqdm

In [23]:
API_KEY = (
    "LK+ayycQ6oz4XLsvmJZHtM3YqV2md7LrcS3sqebIHwn4zri6WAkEPJus9Vn"
    "PZdzMgoJF3pDHZCBjhveBigAgQw=="
)

YEAR             = 2025           # ← change year here
FIRST_MONTH      = f"{YEAR}01"   # January — used for discovery scan
DAILY_LIMIT      = 1_000_000       # API calls per day (set lower if unsure, e.g. 8000)
SAFETY_BUFFER    = 50           # stop this many calls before the hard limit

# ── File paths ─────────────────────────────────────────────────────────────
PARCEL_CSV = r"D:\5-Data\DATA\필지\seoul_merge\parcels_for_energy.csv"
OUTPUT_DIR = r"D:\000_SCI\6_Energy_certificate\0_energy_data\energy_use"

# ── Email notification ──────────────────────────────────────────────────────
# Set SEND_EMAIL = True and fill in your details to receive daily summaries.
#
# Gmail App Password setup (one-time, 2 minutes):
#   1. myaccount.google.com → Security → 2-Step Verification → ON
#   2. Search "App passwords" → Create → name it "energy downloader"
#   3. Copy the 16-character password → paste below as EMAIL_PASSWORD
#
SEND_EMAIL     = True
EMAIL_FROM     = "lina23@hanyang.ac.kr"    # ← your Gmail address
EMAIL_PASSWORD = "Flsk1986**"    # ← 16-char Gmail App Password
EMAIL_TO       = "lina23@hanyang.ac.kr"  # ← where to receive summaries

# ── API settings ───────────────────────────────────────────────────────────
NUM_ROWS  = 100


In [25]:
SEOUL_GU = [
    ("11110", "종로구"),
    ("11140", "중구"),
    ("11170", "용산구"),
    ("11200", "성동구"),
    ("11215", "광진구"),
    ("11230", "동대문구"),
    ("11260", "중랑구"),
    ("11290", "성북구"),
    ("11305", "강북구"),
    ("11320", "도봉구"),
    ("11350", "노원구"),
    ("11380", "은평구"),
    ("11410", "서대문구"),
    ("11440", "마포구"),
    ("11470", "양천구"),
    ("11500", "강서구"),
    ("11530", "구로구"),
    ("11545", "금천구"),
    ("11560", "영등포구"),
    ("11590", "동작구"),
   ("11620", "관악구"),
    ("11650", "서초구"),
    ("11680", "강남구"),
    ("11710", "송파구"),
    ("11740", "강동구"),
]




In [27]:
ENDPOINTS = {
    "electricity": "https://apis.data.go.kr/1613000/BldEngyHubService/getBeElctyUsgInfo",
    "gas":         "https://apis.data.go.kr/1613000/BldEngyHubService/getBeGasUsgInfo",
}

FIRST_MONTH      = f"{YEAR}01"
REMAINING_MONTHS = [f"{YEAR}{m:02d}" for m in range(2, 13)]
ALL_MONTHS       = [f"{YEAR}{m:02d}" for m in range(1, 13)]

os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(
            os.path.join(OUTPUT_DIR, "download.log"), encoding="utf-8"
        ),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger(__name__)

SESSION_STATS = {
    "start_time": datetime.now(),
    "events":     [],
    "errors":     [],
}


# ══════════════════════════════════════════════════════════════════════════
# QUOTA TRACKER — resets automatically each new day
# ══════════════════════════════════════════════════════════════════════════

class QuotaExhausted(Exception):
    pass

class QuotaTracker:
    def __init__(self, label: str):
        self.label = label
        self.limit = DAILY_LIMIT - SAFETY_BUFFER
        self.path  = os.path.join(OUTPUT_DIR, f"quota_{label}.json")
        self.today = str(date.today())
        self._load()

    def _load(self):
        if os.path.exists(self.path):
            data = json.load(open(self.path))
            if data.get("date") == self.today:
                self.count = data["count"]
                return
        self.count = 0   # new day → reset

    def save(self):
        json.dump({"date": self.today, "count": self.count},
                  open(self.path, "w"))

    def increment(self):
        self.count += 1
        if self.count % 100 == 0:
            self.save()

    def exhausted(self) -> bool:
        return self.count >= self.limit

    def status(self) -> str:
        remaining = self.limit - self.count
        return f"{self.count:,} used / {self.limit:,} limit  ({remaining:,} remaining)"


# ══════════════════════════════════════════════════════════════════════════
# API CALL — single paginated request
# ══════════════════════════════════════════════════════════════════════════

def api_get(url: str, params: dict, quota: QuotaTracker) -> list[dict]:
    """Fetch all pages for one query. Raises QuotaExhausted if limit hit."""
    if quota.exhausted():
        raise QuotaExhausted()

    all_items = []
    page      = 1
    total     = None

    while True:
        if quota.exhausted():
            raise QuotaExhausted()

        try:
            r = requests.get(url, params={
                **params,
                "numOfRows":  NUM_ROWS,
                "pageNo":     page,
                "_type":      "json",
                "serviceKey": API_KEY,
            }, timeout=30)
            r.raise_for_status()
        except requests.HTTPError as e:
            log.warning(f"HTTP {e.response.status_code} | {params}")
            quota.increment()
            return all_items
        except Exception as e:
            log.warning(f"Request error: {e}")
            return all_items

        quota.increment()

        try:
            data   = r.json()
            header = data["response"]["header"]
            body   = data["response"]["body"]
        except Exception:
            log.warning(f"Bad JSON: {r.text[:200]}")
            return all_items

        if header.get("resultCode") != "00":
            return all_items

        if total is None:
            total = int(body.get("totalCount", 0))
            if total == 0:
                return all_items

        items = (body.get("items") or {}).get("item", [])
        if isinstance(items, dict):
            items = [items]
        all_items.extend(items)
        time.sleep(DELAY_SEC)

        if len(all_items) >= total or page >= math.ceil(total / NUM_ROWS):
            break
        page += 1

    return all_items


# ══════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════

def append_csv(records: list, path: str):
    if not records:
        return
    pd.DataFrame(records).to_csv(
        path, mode="a",
        header=not os.path.exists(path),
        index=False, encoding="utf-8-sig"
    )

def load_json(path: str) -> dict:
    return json.load(open(path)) if os.path.exists(path) else {}

def save_json(path: str, data: dict):
    json.dump(data, open(path, "w"), ensure_ascii=False)

def log_event(msg: str):
    SESSION_STATS["events"].append(msg)
    log.info(msg)

def log_error(msg: str):
    SESSION_STATS["errors"].append(msg)
    log.error(msg)

def gu_final_path(etype: str, gu_name: str) -> str:
    return os.path.join(OUTPUT_DIR, f"{etype}_{YEAR}_{gu_name}.csv")

def gu_complete(etype: str, gu_name: str) -> bool:
    return os.path.exists(gu_final_path(etype, gu_name))


# ══════════════════════════════════════════════════════════════════════════
# PROCESS ONE GU — both electricity and gas
# ══════════════════════════════════════════════════════════════════════════

def process_gu(gu_cd: str, gu_name: str,
               gu_parcels: pd.DataFrame,
               etype: str,
               quota: QuotaTracker) -> bool:
    """
    For one gu and one energy type:
      A. Scan January → find active parcels
      B. Download Feb–Dec for active parcels
      C. Merge Jan + Feb–Dec → save final CSV

    Returns True if complete, False if quota exhausted mid-way.
    """
    url         = ENDPOINTS[etype]
    out_dir_gu  = os.path.join(OUTPUT_DIR, "gu_progress")
    os.makedirs(out_dir_gu, exist_ok=True)

    active_path = os.path.join(out_dir_gu, f"active_{etype}_{gu_name}.csv")
    ckpt_path   = os.path.join(out_dir_gu, f"ckpt_{etype}_{gu_name}.csv")
    prog_path   = os.path.join(out_dir_gu, f"prog_{etype}_{gu_name}.json")
    final_path  = gu_final_path(etype, gu_name)

    if os.path.exists(final_path):
        n = len(pd.read_csv(final_path))
        log.info(f"  ✓ {gu_name} {etype} already done ({n:,} records)")
        return True

    prog = load_json(prog_path)

    # ── Step A: January scan ─────────────────────────────────────
    if not prog.get("jan_scan_complete"):
        start_idx = prog.get("jan_scan_idx", 0)

        log.info(f"  [{gu_name}] {etype} — January scan "
                 f"({len(gu_parcels):,} parcels, from index {start_idx:,})")

        active_buf = []

        pbar = tqdm(
            total=len(gu_parcels), initial=start_idx,
            desc=f"  {gu_name[:4]} Jan {etype[:5]}", unit="parcel", ncols=80,
        )

        try:
            for idx in range(start_idx, len(gu_parcels)):
                row   = gu_parcels.iloc[idx]
                items = api_get(url, {
                    "sigunguCd": row["sigunguCd"],
                    "bjdongCd":  row["bjdongCd"],
                    "bun":       row["bun"],
                    "ji":        row["ji"],
                    "useYm":     FIRST_MONTH,
                }, quota)

                if items:
                    active_buf.append({
                        "sigunguCd":  row["sigunguCd"],
                        "bjdongCd":   row["bjdongCd"],
                        "bun":        row["bun"],
                        "ji":         row["ji"],
                        "guName":     gu_name,
                        "PNU":        row.get("PNU", ""),
                        "platPlc":    items[0].get("platPlc", ""),
                        "useQty_jan": items[0].get("useQty", ""),
                    })

                pbar.update(1)
                if len(active_buf) >= 200:
                    append_csv(active_buf, active_path)
                    active_buf = []

        except QuotaExhausted:
            pbar.close()
            if active_buf:
                append_csv(active_buf, active_path)
            prog["jan_scan_idx"] = idx
            save_json(prog_path, prog)
            quota.save()
            n_active = len(pd.read_csv(active_path)) if os.path.exists(active_path) else 0
            log_event(
                f"QUOTA: {gu_name} {etype} Jan scan — "
                f"{idx:,}/{len(gu_parcels):,} done, {n_active:,} active found"
            )
            return False   # quota exhausted

        pbar.close()
        if active_buf:
            append_csv(active_buf, active_path)

        prog["jan_scan_complete"] = True
        prog["jan_scan_idx"]      = len(gu_parcels)
        save_json(prog_path, prog)

        n_active = len(pd.read_csv(active_path)) if os.path.exists(active_path) else 0
        log.info(f"  [{gu_name}] {etype} Jan scan done — {n_active:,} active parcels")

    # ── Step B: Download Feb–Dec ──────────────────────────────────
    if not prog.get("remaining_complete"):
        if not os.path.exists(active_path):
            log.info(f"  [{gu_name}] {etype} — no active parcels, skipping")
            prog["remaining_complete"] = True
            save_json(prog_path, prog)
        else:
            active = pd.read_csv(active_path, dtype=str)
            for col, n in {"sigunguCd":5,"bjdongCd":5,"bun":4,"ji":4}.items():
                active[col] = active[col].str.zfill(n)

            if len(active) == 0:
                prog["remaining_complete"] = True
                save_json(prog_path, prog)
            else:
                # Resume: load done set
                done: set = set()
                if os.path.exists(ckpt_path):
                    ck   = pd.read_csv(ckpt_path, dtype=str)
                    done = set(zip(ck["q_sigunguCd"], ck["q_bjdongCd"],
                                   ck["q_bun"],       ck["q_ji"], ck["q_useYm"]))

                n_total = len(active) * len(REMAINING_MONTHS)
                log.info(
                    f"  [{gu_name}] {etype} — Feb–Dec download "
                    f"({len(active):,} parcels × {len(REMAINING_MONTHS)} months, "
                    f"{len(done):,} already done)"
                )

                records = []
                pbar = tqdm(
                    total=n_total, initial=len(done),
                    desc=f"  {gu_name[:4]} Feb-Dec {etype[:5]}", unit="req", ncols=80,
                )

                try:
                    for _, row in active.iterrows():
                        sgg = row["sigunguCd"]
                        bjd = row["bjdongCd"]
                        bun = row["bun"]
                        ji  = row["ji"]

                        for ym in REMAINING_MONTHS:
                            pbar.update(1)
                            if (sgg, bjd, bun, ji, ym) in done:
                                continue

                            items = api_get(url, {
                                "sigunguCd": sgg, "bjdongCd": bjd,
                                "bun": bun, "ji": ji, "useYm": ym,
                            }, quota)

                            for item in items:
                                item.update({
                                    "energy_type": etype,
                                    "q_sigunguCd": sgg, "q_bjdongCd": bjd,
                                    "q_bun":       bun, "q_ji":       ji,
                                    "q_useYm":     ym,
                                    "guName":      gu_name,
                                    "PNU":         row.get("PNU", ""),
                                })
                                records.append(item)

                            if len(records) >= 2_000:
                                append_csv(records, ckpt_path)
                                records = []

                except QuotaExhausted:
                    pbar.close()
                    if records:
                        append_csv(records, ckpt_path)
                    prog["jan_scan_complete"] = True   # keep Jan scan result
                    save_json(prog_path, prog)
                    quota.save()
                    n_done = len(done) + quota.count
                    log_event(
                        f"QUOTA: {gu_name} {etype} Feb–Dec — "
                        f"{n_done:,}/{n_total:,} done"
                    )
                    return False   # quota exhausted

                pbar.close()
                if records:
                    append_csv(records, ckpt_path)

                prog["remaining_complete"] = True
                save_json(prog_path, prog)

    # ── Step C: Merge Jan + Feb–Dec → final CSV ───────────────────
    log.info(f"  [{gu_name}] {etype} — merging all months...")

    frames = []

    # January records (from active parcel list)
    if os.path.exists(active_path):
        jan = pd.read_csv(active_path, dtype=str)
        jan = jan.rename(columns={"useQty_jan": "useQty"})
        jan["energy_type"] = etype
        jan["useYm"]       = FIRST_MONTH
        jan["q_sigunguCd"] = jan["sigunguCd"]
        jan["q_bjdongCd"]  = jan["bjdongCd"]
        jan["q_bun"]       = jan["bun"]
        jan["q_ji"]        = jan["ji"]
        jan["q_useYm"]     = FIRST_MONTH
        frames.append(jan)

    # Feb–Dec records
    if os.path.exists(ckpt_path):
        frames.append(pd.read_csv(ckpt_path, dtype=str))

    if frames:
        final = pd.concat(frames, ignore_index=True)
        final.drop_duplicates(
            subset=["q_sigunguCd","q_bjdongCd","q_bun","q_ji","q_useYm"],
            inplace=True
        )
        final.to_csv(final_path, index=False, encoding="utf-8-sig")

        # Clean up intermediate files
        for f in [active_path, ckpt_path, prog_path]:
            if os.path.exists(f):
                os.remove(f)

        log_event(f"✓ {gu_name} {etype} complete — {len(final):,} records")
        log.info(f"  ✓ [{gu_name}] {etype} saved → {final_path}  ({len(final):,} records)")
    else:
        log.info(f"  [{gu_name}] {etype} — no data found")
        # Mark as done with empty file
        pd.DataFrame().to_csv(final_path, index=False, encoding="utf-8-sig")

    return True   # complete



# ══════════════════════════════════════════════════════════════════════════
# EMAIL SUMMARY
# ══════════════════════════════════════════════════════════════════════════

def send_summary_email(subject_suffix: str = ""):
    if not SEND_EMAIL:
        return
    try:
        elapsed = datetime.now() - SESSION_STATS["start_time"]
        h, rem  = divmod(int(elapsed.total_seconds()), 3600)
        m, s    = divmod(rem, 60)

        # Gu completion status
        gu_lines = ""
        for gu_cd, gu_name in SEOUL_GU:
            elec_done = gu_complete("electricity", gu_name)
            gas_done  = gu_complete("gas", gu_name)
            elec_sym  = "✓" if elec_done else "—"
            gas_sym   = "✓" if gas_done  else "—"
            gu_lines += f"  {gu_name:<6}  전기:{elec_sym}  가스:{gas_sym}\n"

        elec_done_count = sum(1 for _, g in SEOUL_GU if gu_complete("electricity", g))
        gas_done_count  = sum(1 for _, g in SEOUL_GU if gu_complete("gas", g))

        quota = QuotaTracker("main")

        body = (
            f"Seoul Building Energy Downloader — Daily Summary\n"
            f"{'='*52}\n"
            f"Date      : {date.today()}\n"
            f"Duration  : {h}h {m}m {s}s\n"
            f"Year      : {YEAR}\n"
            f"Output    : {OUTPUT_DIR}\n"
            f"\nPROGRESS\n"
            f"  전기: {elec_done_count}/25 구 완료\n"
            f"  가스: {gas_done_count}/25 구 완료\n"
            f"  API calls today: {quota.count:,} / {quota.limit:,}\n"
            f"\nGU STATUS\n{gu_lines}"
            f"\nSESSION EVENTS\n"
            f"{''.join(f'  • {e}{chr(10)}' for e in SESSION_STATS['events']) or '  (none)'}\n"
            + (f"\nERRORS\n{''.join(f'  ⚠ {e}{chr(10)}' for e in SESSION_STATS['errors'])}\n"
               if SESSION_STATS["errors"] else "")
            + f"\n{'='*52}\n"
            f"Run the script again tomorrow to continue.\n"
        )

        msg = MIMEMultipart()
        msg["From"]    = EMAIL_FROM
        msg["To"]      = EMAIL_TO
        msg["Subject"] = f"[Energy Download] {date.today()} — {subject_suffix}"
        msg.attach(MIMEText(body, "plain", "utf-8"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(EMAIL_FROM, EMAIL_PASSWORD)
            server.sendmail(EMAIL_FROM, EMAIL_TO, msg.as_string())

        log.info(f"✉  Summary email sent → {EMAIL_TO}")
    except Exception as e:
        log.warning(f"Email failed: {e}")



In [29]:
if __name__ == "__main__":

    log.info("═" * 65)
    log.info("  Seoul Building Energy Downloader — Gu by Gu")
    log.info(f"  Date   : {date.today()}")
    log.info(f"  Year   : {YEAR}")
    log.info(f"  Output : {OUTPUT_DIR}")
    log.info(f"  Email  : {'enabled → ' + EMAIL_TO if SEND_EMAIL else 'disabled'}")
    log.info("═" * 65)

    try:
        # Load all parcels
        if not os.path.exists(PARCEL_CSV):
            log_error(f"Parcel file not found: {PARCEL_CSV}")
            sys.exit(1)

        all_parcels = pd.read_csv(PARCEL_CSV, dtype=str)
        for col, n in {"sigunguCd":5,"bjdongCd":5,"bun":4,"ji":4}.items():
            all_parcels[col] = all_parcels[col].str.zfill(n)
        for col in ("guName","PNU","platPlc"):
            if col not in all_parcels.columns:
                all_parcels[col] = ""

        log.info(f"Loaded {len(all_parcels):,} total parcels")

        # Shared quota tracker for this session
        quota = QuotaTracker("main")
        log.info(f"Quota today: {quota.status()}")

        # Process each gu in order
        for gu_cd, gu_name in SEOUL_GU:

            # Filter parcels for this gu
            gu_parcels = all_parcels[
                all_parcels["sigunguCd"] == gu_cd
            ].reset_index(drop=True)

            log.info(f"\n{'═'*65}")
            log.info(f"  구: {gu_name} ({gu_cd})  —  {len(gu_parcels):,} parcels")
            log.info(f"  Quota: {quota.status()}")
            log.info(f"{'═'*65}")

            if len(gu_parcels) == 0:
                log.info(f"  No parcels found for {gu_name} — skipping")
                continue

            # ── Electricity ───────────────────────────────────────
            elec_done = gu_complete("electricity", gu_name)
            if not elec_done:
                done = process_gu(gu_cd, gu_name, gu_parcels,
                                  "electricity", quota)
                if not done:
                    # Quota exhausted mid-gu
                    log.info(f"\n{'═'*65}")
                    log.info(f"  ⏸  QUOTA REACHED — stopping for today")
                    log.info(f"  Last gu: {gu_name} (electricity)")
                    log.info(f"  ▶  Run again tomorrow to continue")
                    log.info(f"{'═'*65}")
                    send_summary_email(subject_suffix=f"⏸ Stopped at {gu_name} electricity")
                    sys.exit(0)
            else:
                log.info(f"  ✓ {gu_name} electricity already complete")

            # ── Gas ───────────────────────────────────────────────
            gas_done = gu_complete("gas", gu_name)
            if not gas_done:
                done = process_gu(gu_cd, gu_name, gu_parcels,
                                  "gas", quota)
                if not done:
                    log.info(f"\n{'═'*65}")
                    log.info(f"  ⏸  QUOTA REACHED — stopping for today")
                    log.info(f"  Last gu: {gu_name} (gas)")
                    log.info(f"  ▶  Run again tomorrow to continue")
                    log.info(f"{'═'*65}")
                    send_summary_email(subject_suffix=f"⏸ Stopped at {gu_name} gas")
                    sys.exit(0)
            else:
                log.info(f"  ✓ {gu_name} gas already complete")

        log_event("✓ ALL COMPLETE — Seoul electricity + gas downloaded")

    except Exception as e:
        log_error(f"Unexpected error: {e}\n{traceback.format_exc()}")
        send_summary_email(subject_suffix="❌ ERROR")
        raise

    # Final file list
    log.info(f"\n{'═'*65}  Output files:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if f.endswith(".csv"):
            sz = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1_048_576
            log.info(f"  {f:<52}  {sz:6.2f} MB")

    send_summary_email(subject_suffix="✓ All complete!")

2026-03-09 19:26:08,710 [INFO] ═════════════════════════════════════════════════════════════════
2026-03-09 19:26:08,712 [INFO]   Seoul Building Energy Downloader — Gu by Gu
2026-03-09 19:26:08,712 [INFO]   Date   : 2026-03-09
2026-03-09 19:26:08,713 [INFO]   Year   : 2025
2026-03-09 19:26:08,714 [INFO]   Output : D:\000_SCI\6_Energy_certificate\0_energy_data\energy_use
2026-03-09 19:26:08,715 [INFO]   Email  : enabled → lina23@hanyang.ac.kr
2026-03-09 19:26:08,716 [INFO] ═════════════════════════════════════════════════════════════════
2026-03-09 19:26:10,421 [INFO] Loaded 902,001 total parcels
2026-03-09 19:26:10,423 [INFO] Quota today: 66,300 used / 999,950 limit  (933,650 remaining)
2026-03-09 19:26:10,475 [INFO] 
═════════════════════════════════════════════════════════════════
2026-03-09 19:26:10,477 [INFO]   구: 종로구 (11110)  —  49,087 parcels
2026-03-09 19:26:10,477 [INFO]   Quota: 66,300 used / 999,950 limit  (933,650 remaining)
2026-03-09 19:26:10,478 [INFO] ═══════════════════

  종로구 Feb-Dec elect:  13%|█▌          | 12023/94138 [00:02<25:21, 53.98req/s]

2026-03-09 19:26:12,844 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0007', 'ji': '0018', 'useYm': '202508'}
2026-03-09 19:26:12,871 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0007', 'ji': '0018', 'useYm': '202509'}
2026-03-09 19:26:12,900 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0007', 'ji': '0018', 'useYm': '202510'}


  종로구 Feb-Dec elect:  13%|█▌          | 12074/94138 [00:03<22:46, 60.08req/s]

2026-03-09 19:26:13,721 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0030', 'ji': '0007', 'useYm': '202511'}
2026-03-09 19:26:13,747 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0030', 'ji': '0007', 'useYm': '202512'}
2026-03-09 19:26:13,783 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0030', 'ji': '0015', 'useYm': '202511'}
2026-03-09 19:26:13,818 [WARNING] HTTP 429 | {'sigunguCd': '11110', 'bjdongCd': '10100', 'bun': '0030', 'ji': '0015', 'useYm': '202512'}


...

2026-03-10 12:40:54,861 [WARNING] HTTP 502 | {'sigunguCd': '11140', 'bjdongCd': '13600', 'bun': '0023', 'ji': '0009', 'useYm': '202510'}


  중구 Feb-Dec elect:  36%|███▌      | 28135/79222 [2:49:37<4:54:35,  2.89req/s]

2026-03-10 13:06:48,853 [WARNING] HTTP 502 | {'sigunguCd': '11140', 'bjdongCd': '14300', 'bun': '0035', 'ji': '0002', 'useYm': '202509'}


  중구 Feb-Dec elect:  38%|███▊      | 30433/79222 [3:00:57<2:16:00,  5.98req/s]

KeyboardInterrupt: 